In [ ]:
{
 "cells": [
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "# 00 — Environment Setup and Holdout Isolation"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "from google.colab import drive\n",
    "drive.mount('/content/drive')"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "import os\n",
    "from pathlib import Path\n",
    "\n",
    "REPO_URL = 'https://github.com/<org>/vuln-detection.git'\n",
    "REPO_DIR = '/content/vuln-detection'\n",
    "\n",
    "if not os.path.isdir(REPO_DIR):\n",
    "    !git clone {REPO_URL} {REPO_DIR}\n",
    "else:\n",
    "    !git -C {REPO_DIR} pull origin main\n",
    "\n",
    "%cd {REPO_DIR}\n",
    "\n",
    "!pip install -q -e .\n",
    "!pip install -q transformers pyreft peft accelerate layer_to_layer"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "import shutil\n",
    "from pathlib import Path\n",
    "\n",
    "DRIVE_DIR = Path('/content/drive/MyDrive/vuln-detection-data/')\n",
    "RAW_DATA_TARGET_DIR = Path('data/raw')\n",
    "RAW_DATA_TARGET_DIR.mkdir(parents=True, exist_ok=True)\n",
    "\n",
    "if DRIVE_DIR.exists():\n",
    "    candidates = list(DRIVE_DIR.glob('*.jsonl')) + list(DRIVE_DIR.glob('*.json')) + list(DRIVE_DIR.glob('*.csv'))\n",
    "    if candidates:\n",
    "        source_file = candidates[0]\n",
    "        target_file_path = RAW_DATA_TARGET_DIR / source_file.name\n",
    "        if not target_file_path.exists():\n",
    "            shutil.copy(source_file, target_file_path)\n",
    "            print(f'Staged dataset: {target_file_path}')\n",
    "    else:\n",
    "        print('No valid dataset file found in Drive folder.')"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "from src.data import load_rdiversevul, make_holdout_split, load_split_config\n",
    "\n",
    "df = load_rdiversevul(data_dir='data/raw', drop_duplicates=True)\n",
    "split_params = load_split_config(config_path='configs/track_a.yaml')\n",
    "\n",
    "try:\n",
    "    train_idx, holdout_idx = make_holdout_split(\n",
    "        df=df,\n",
    "        test_size=split_params['test_size'],\n",
    "        random_state=split_params['random_state'],\n",
    "        overwrite=False\n",
    "    )\n",
    "except FileExistsError:\n",
    "    print('Split files already exist. Skipping split creation.')"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "import torch\n",
    "assert torch.cuda.is_available(), 'CRITICAL: GPU acceleration required for Track B.'"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "from transformers import AutoConfig, AutoTokenizer\n",
    "from src.utils import load_config\n",
    "\n",
    "cfg_b = load_config('configs/track_b.yaml')\n",
    "AutoTokenizer.from_pretrained(cfg_b.model.name, trust_remote_code=True)\n",
    "AutoConfig.from_pretrained(cfg_b.model.name, trust_remote_code=True)"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "import platform\n",
    "import psutil\n",
    "from src.data import load_holdout_split\n",
    "\n",
    "train_idx, holdout_idx = load_holdout_split()\n",
    "print(f'Python: {platform.python_version()}')\n",
    "print(f'PyTorch: {torch.__version__}')\n",
    "print(f'RAM: {psutil.virtual_memory().total / 1e9:.2f} GB')\n",
    "print(f'Train size: {len(train_idx)} | Holdout size: {len(holdout_idx)}')"
   ]
  }
 ],
 "metadata": {
  "kernelspec": { "display_name": "Python 3", "language": "python", "name": "python3" },
  "language_info": { "name": "python", "version": "3.10.0" }
 },
 "nbformat": 4,
 "nbformat_minor": 5
}